# XGBoost Experiment (Puripat, wk2–3): ตัวทดลอง

ยังไม่ใช่ผลสุดท้าย เป็นร่างสำหรับเทียบว่าฟีเจอร์ใหม่จาก `feature_pipeline.py` ช่วยจริงไหม
ก่อนตัดสินใจเลือก run ร่วมกับ Saphondanai และก่อน promote เข้า MLflow registry

เทียบ 2 ชุด: base (ผลจาก `clean_data`) กับ +features (เพิ่มฟีเจอร์ 7 ตัว)
log ลง MLflow ในเครื่อง (`../mlflow.db` SQLite) ไปก่อน แล้วย้ายไป URI กลางเมื่อ Saphondanai ตั้งเสร็จ

In [1]:
import sys
sys.path.append("../src")
import mlflow
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from xgboost import XGBClassifier

from clean_pipeline import RAW_FILENAME, clean_data, load_raw_data
from feature_pipeline import add_features

mlflow.set_tracking_uri("sqlite:///../mlflow.db")  # ponytail: local store, สลับเป็น URI กลางของทีมเมื่อพร้อม
mlflow.set_experiment("attrition-xgboost-P-experiment")

base = clean_data(load_raw_data(f"../data/raw/{RAW_FILENAME}"))
sets = {"base": base, "base+features": add_features(base)}
{k: v.shape for k, v in sets.items()}

c:\Users\HP\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'base': (1470, 48), 'base+features': (1470, 55)}

## เทรน + ประเมิน

stratified split 80/20 (seed 42), `scale_pos_weight` แก้ imbalance 16%, ประเมินด้วย AUC / PR-AUC / F1 (threshold 0.5)

In [2]:
PARAMS = dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)

def run(name, df):
    X, y = df.drop(columns="Attrition"), df["Attrition"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    model = XGBClassifier(**PARAMS, scale_pos_weight=spw, eval_metric="logloss")
    with mlflow.start_run(run_name=name):
        model.fit(Xtr, ytr)
        p = model.predict_proba(Xte)[:, 1]
        m = {"auc": roc_auc_score(yte, p), "pr_auc": average_precision_score(yte, p), "f1": f1_score(yte, p > 0.5)}
        mlflow.log_params({**PARAMS, "scale_pos_weight": round(spw, 2), "n_features": X.shape[1], "feature_set": name})
        mlflow.log_metrics(m)
    return m, model, X

results, models = {}, {}
for name, df in sets.items():
    results[name], models[name], X = run(name, df)
pd.DataFrame(results).T.round(3)

,auc,pr_auc,f1
base,0.788,0.520,0.469
base+features,0.781,0.527,0.500


ชุด test มีแค่ ~294 แถว (positive ~47) ค่า AUC ผันผวนตาม split ได้พอสมควร
ถ้าผลต่างระหว่างสองชุดเล็ก ให้ดูด้วย cross-validation ด้านล่างก่อนสรุป

In [3]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = {}
for name, df in sets.items():
    X, y = df.drop(columns="Attrition"), df["Attrition"]
    spw = (y == 0).sum() / (y == 1).sum()
    p = cross_val_predict(XGBClassifier(**PARAMS, scale_pos_weight=spw), X, y, cv=cv, method="predict_proba")[:, 1]
    rows[name] = {"cv_auc": roc_auc_score(y, p), "cv_pr_auc": average_precision_score(y, p)}
pd.DataFrame(rows).T.round(3)

,cv_auc,cv_pr_auc
base,0.815,0.579
base+features,0.803,0.573


## ฟีเจอร์ที่โมเดลใช้มากที่สุด (ชุด +features)

In [4]:
m = models["base+features"]
pd.Series(m.feature_importances_, index=m.feature_names_in_).sort_values(ascending=False).head(15).round(3)

OverTime                         0.047
JobHopping                       0.037
JobLevel                         0.036
JobRole_Sales Executive          0.033
StockOptionLevel                 0.033
MaritalStatus_Single             0.030
OverTimeXDistance                0.027
Department_Sales                 0.027
AvgSatisfaction                  0.027
TotalWorkingYears                0.026
JobRole_Laboratory Technician    0.025
YearsAtCompany                   0.024
ManagerStability                 0.023
Age                              0.023
EnvironmentSatisfaction          0.022
dtype: float32